In [1]:
import os 
import time 
import logging
import requests 
import pandas 
from datetime import datetime 
from pathlib import Path 
from requests.adapters import HTTPAdapter 
from urllib3.util.retry import Retry

In [2]:
try:
    from dotenv import load_dotenv
except ImportError:
    pass

In [3]:
API_KEY = os.getenv("OPENWEATHER_API_KEY","")
BASE_URL = "https://api.openweathermap.org/data/2.5"
OUTPUT_DIR = Path("Downloads")
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
CITIES = [
    "Nagpur,IN",
    "Mumbai,IN",
    "Delhi,IN",
    "Bangalore,IN",
    "London,GB",
]

logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s]%(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(OUTPUT_DIR/"pipeline.log"),
    ]
)
log=logging.getLogger(__name__)

In [5]:
def build_session():
    session=requests.Session()
    retry=Retry(
        total = 3,
        backoff_factor = 1,
        status_forcelist=(429,500,502,503,504),
        allowed_methods = ["GET"],
    )
    session.mount("https://",HTTPAdapter(max_retries=retry))
    return session

In [7]:
 # step 2 fetch current wather for one city
 
def fetch_current_weather(session,city:str)-> dict | None:
    """Calls OpenWeather current weather endpoint.
    Returns raw JSON dict or None on failure."""
    try: 
        response=session.get(
            f"{BASE_URL}/weather",
            params={
                "q":city,
                "appid":API_KEY,
                "units":"metric",
            },
            timeout=(5,15)
        )
        response.raise_for_status()
        log.info(f" {city}: {response.status_code}")
        return response.json()
    except requests.exceptions.HTTPError as e:
        status=e.response.status_code
        if status == 400:
            log.error("invalid API key - check API kEY")
        elif status == 404:
            log.warning(f"city not found: {city}")
        elif status==429:
            log.warning("rate limited - waiting 60s")
            time.sleep(60)
        else:
            log.error(f"HTTP {status} for {city} : {e}")
        return None
    
    except requests.exceptions.Timeout:
        log.warning(f"Timeout for {city}")
        return None
    
    except Exception as e:
        log.error(f"Unexcpected error for {city}:{e}")
        return None

20
10
